# Job search agent: 1 agent, 3 tools, Claude + LangSmith

The class exercise (one agent, three tools, fall back to the model's own knowledge, trace everything in LangSmith), applied to a CV and LinkedIn job-search service instead of weather, forex and news.

| Lecture tool | This notebook | Source |
|---|---|---|
| `get_weather` | `search_jobs`: live UK job ads | Adzuna search API |
| `get_forex` | `get_salary_spread`: how pay is spread for a role and location | Adzuna histogram API |
| `get_news` | `get_candidate_profile`: the job seeker's facts | Local JSON (fictional sample) |
| LLM brain | CV wording, cover letters, LinkedIn advice, general career questions | Claude, no tool |

The model is Claude (`ChatAnthropic`) instead of Grok. The candidate in this notebook is **fictional**. Don't put real client data in a notebook you'll share.

## 1. Install

In [ ]:
%pip install -q langchain langchain-anthropic langgraph langsmith python-dotenv requests

## 2. Keys and tracing

Put these in a `.env` file next to the notebook, and keep `.env` out of git:

```
ANTHROPIC_API_KEY=...
ADZUNA_APP_ID=...
ADZUNA_APP_KEY=...
LANGSMITH_API_KEY=...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=job-search-agent
```

If a key is missing, the cell asks for it instead. With `LANGSMITH_TRACING=true`, every agent run appears in the LangSmith dashboard under the project name, showing each model call and tool call.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

for key in ["ANTHROPIC_API_KEY", "ADZUNA_APP_ID", "ADZUNA_APP_KEY", "LANGSMITH_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "job-search-agent")
print("Tracing to LangSmith project:", os.environ["LANGSMITH_PROJECT"])

## 3. The candidate profile

This stands in for stage 1 of the service (extracting facts from the CV and intake form). It's a fictional person.

In [ ]:
import json

CANDIDATE_PROFILE = {
    "name": "Alex Morgan (fictional)",
    "location": "London",
    "willing_to_commute": "Up to 60 minutes; open to hybrid",
    "target_roles": ["Project Coordinator", "Project Officer", "Resident Liaison Officer"],
    "min_salary_gbp": 32000,
    "experience": [
        {"title": "Office Coordinator", "employer": "Housing association (anonymised)", "years": 3,
         "highlights": ["Scheduled works for 400+ homes", "First contact for resident complaints"]},
        {"title": "Customer Service Adviser", "employer": "Energy supplier (anonymised)", "years": 2,
         "highlights": ["Handled 60+ calls a day", "Trained 5 new starters"]},
    ],
    "qualifications": ["PRINCE2 Foundation", "Full UK driving licence"],
    "dealbreakers": ["No IT or software project roles", "No senior or head-of roles yet"],
}

## 4. The three tools

The docstrings are what the agent reads to decide which tool to call, so they say clearly what each tool is *for* and what it is *not* for.

In [ ]:
import requests
from langchain.tools import tool

ADZUNA = "https://api.adzuna.com/v1/api/jobs/gb"

def _adzuna(path: str, params: dict) -> dict:
    params = {"app_id": os.environ["ADZUNA_APP_ID"], "app_key": os.environ["ADZUNA_APP_KEY"], **params}
    r = requests.get(f"{ADZUNA}/{path}", params=params, timeout=20)
    r.raise_for_status()
    return r.json()


@tool
def search_jobs(what: str, where: str = "London", max_results: int = 5) -> str:
    """Search live UK job ads on Adzuna.
    Use when the user wants to find, list or compare actual job openings.
    `what` is the job title or keywords, `where` a UK town, city or region.
    Descriptions are short snippets, not full job descriptions."""
    try:
        data = _adzuna("search/1", {"what": what, "where": where,
                                    "results_per_page": min(max_results, 10), "sort_by": "date"})
    except requests.RequestException as e:
        return f"Job search failed ({e}). Tell the user the job board is unavailable right now."
    jobs = [{
        "title": j.get("title"),
        "company": j.get("company", {}).get("display_name"),
        "location": j.get("location", {}).get("display_name"),
        "salary_min": j.get("salary_min"),
        "salary_max": j.get("salary_max"),
        "posted": j.get("created", "")[:10],
        "snippet": j.get("description", "")[:300],
        "url": j.get("redirect_url"),
    } for j in data.get("results", [])]
    return json.dumps({"total_matching_ads": data.get("count"), "jobs": jobs, "source": "Jobs by Adzuna"})


@tool
def get_salary_spread(what: str, where: str = "London") -> str:
    """Get how advertised salaries are spread for a job title in a UK location (Adzuna).
    Use for questions about pay: what a role pays, whether a salary is fair, what to ask for.
    Returns a count of ads per salary band; it is not a list of jobs."""
    try:
        data = _adzuna("histogram", {"what": what, "where": where})
    except requests.RequestException as e:
        return f"Salary lookup failed ({e}). Tell the user salary data is unavailable right now."
    bands = {int(k): v for k, v in data.get("histogram", {}).items()}
    total = sum(bands.values())
    if not total:
        return f"No salary data found for '{what}' in {where}."
    return json.dumps({"role": what, "location": where, "ads_with_salary": total,
                       "ads_per_salary_band_gbp": dict(sorted(bands.items())), "source": "Adzuna"})


@tool
def get_candidate_profile() -> str:
    """Get the job seeker's own facts: experience, qualifications, target roles,
    minimum salary, location and dealbreakers.
    Use whenever an answer depends on who this candidate is (fit, CV wording, cover letters).
    Never invent experience that is not in this profile."""
    return json.dumps(CANDIDATE_PROFILE)


tools = [search_jobs, get_salary_spread, get_candidate_profile]

Test each tool on its own before handing it to the agent. These calls use Adzuna's free quota (250 a day) but no Claude credit.

In [ ]:
print(search_jobs.invoke({"what": "project coordinator", "where": "London", "max_results": 2})[:800])
print()
print(get_salary_spread.invoke({"what": "project coordinator", "where": "London"}))

## 5. The model and the agent

`create_agent` builds the standard tool-calling loop on LangGraph: the model decides whether to call a tool, sees the result, and repeats until it can answer. If no tool fits, it answers from its own knowledge, which is the "LLM brain" fallback.

The model is a variable, so you can compare models on the same questions in LangSmith. Claude 5.5 models think by default, so thinking is switched off here to keep runs fast and cheap. Turn it on if you want to see reasoning in the traces.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent

MODEL = "claude-sonnet-5-5"   # or "claude-haiku-5-5" (much cheaper) / "claude-opus-5-5"

llm = ChatAnthropic(model=MODEL, max_tokens=2000, temperature=0,
                    thinking={"type": "disabled"})

SYSTEM_PROMPT = """You are a UK job search assistant for one job seeker.

Tools:
- search_jobs: live job ads. Use for finding or listing openings.
- get_salary_spread: pay data. Use for salary questions.
- get_candidate_profile: the job seeker's facts. Use whenever the answer depends on who they are.

Rules:
- If the question needs none of these (general career advice, LinkedIn tips, interview questions,
  anything off-topic), answer from your own knowledge without calling a tool.
- Never invent experience, qualifications or numbers. If the profile doesn't say it, ask.
- When you list jobs, include the link and say they come from Adzuna.
- Write in plain UK English. Keep answers short."""

agent = create_agent(llm, tools=tools, system_prompt=SYSTEM_PROMPT, name="job_search_agent")

## 6. Ask questions

`ask()` prints which tools the agent chose, then the answer. The `run_name` and `tags` make each question easy to find in LangSmith.

In [ ]:
def ask(question: str, tag: str = "demo"):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"run_name": f"job-agent: {question[:40]}", "tags": [tag, MODEL]},
    )
    used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
    print(f"Q: {question}\nTools used: {used or 'none (LLM brain)'}\n")
    print(result["messages"][-1].content)
    print("\n" + "-" * 80 + "\n")
    return result

In [ ]:
ask("Find me 3 recent project coordinator jobs in Manchester.", tag="tool:search_jobs");

In [ ]:
ask("What does a resident liaison officer usually get paid in London?", tag="tool:salary");

In [ ]:
ask("What experience do I have that's relevant to resident liaison work?", tag="tool:profile");

In [ ]:
ask("Find project officer roles in London that pay at least my minimum, and tell me which fits me best.",
    tag="multi-tool");

In [ ]:
ask("Write me a LinkedIn headline, under 200 characters.", tag="profile-or-brain");

In [ ]:
ask("What's the difference between a project coordinator and a project manager?", tag="llm-brain");

## 7. What to look for in LangSmith

Open the `job-search-agent` project at https://smith.langchain.com. For each run, check:

- **Routing:** did each question go to the tool you expected? The tags in the cells above say which one should have been used.
- **The LLM-brain question** should show one model call and no tool calls.
- **The multi-tool question** should show the profile call *and* the job search, then one answer that uses both.
- **Cost and latency** per run, and how they change when you set `MODEL = "claude-haiku-5-5"` and re-run.

## Where this goes next

- **Multi-agent (CrewAI or LangGraph supervisor):** split the service's stages into agents, for example a fact extractor, a CV writer and a quality checker that sends work back.
- **A fourth tool:** `tailor_cv_for_job(url)`, which is the "adjust my CV from the listing" feature.
- **Evaluation:** turn the questions above into a LangSmith dataset with the expected tool for each, so you can measure routing accuracy when you change the prompt or model.